# 04 · Aggregations

Summarising data: totals, averages, counts per group, pivots. If you know SQL `GROUP BY`, this will feel familiar.

In [1]:
# Setup — run this cell first every time you open the notebook
import os, sys
os.environ.setdefault("JAVA_HOME", "/opt/homebrew/opt/openjdk@17")   # Spark runs on Java
os.environ["PYSPARK_PYTHON"] = sys.executable                         # workers use this same Python

from pyspark.sql import SparkSession, Window, functions as F, types as T

spark = (SparkSession.builder
         .appName("pyspark-course")
         .master("local[*]")                               # run locally, use all CPU cores
         .config("spark.sql.shuffle.partitions", "8")      # explained in lesson 10
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("FATAL")                   # keep Java's log noise out of the notebook

DATA, OUT = "../data", "../output"

In [2]:
orders = spark.read.csv(f"{DATA}/orders.csv", header=True, inferSchema=True)
customers = spark.read.csv(f"{DATA}/customers.csv", header=True, inferSchema=True)

## Aggregating a whole table

In [3]:
orders.agg(
    F.count("*").alias("n_orders"),
    F.sum("quantity").alias("total_qty"),
    F.round(F.avg("quantity"), 3).alias("avg_qty"),
    F.min("order_ts").alias("first"),
    F.max("order_ts").alias("last"),
    F.countDistinct("customer_id").alias("n_customers"),
).show(truncate=False)

+--------+---------+-------+-------------------+-------------------+-----------+
|n_orders|total_qty|avg_qty|first              |last               |n_customers|
+--------+---------+-------+-------------------+-------------------+-----------+
|5000    |11081    |2.216  |2024-01-01 00:19:58|2024-12-31 23:49:57|180        |
+--------+---------+-------+-------------------+-------------------+-----------+



### `count("*")` vs `count(col)`
`count("*")` counts rows. `count("col")` counts **non-null** values in that column.

In [4]:
customers.agg(F.count("*").alias("rows"), F.count("country").alias("with_country")).show()

+----+------------+
|rows|with_country|
+----+------------+
| 200|         185|
+----+------------+



## `groupBy` + `agg`
`groupBy(cols)` splits rows into groups; `agg(...)` computes one row per group.

In [5]:
orders.groupBy("status").count().show()

(orders
 .groupBy("status")
 .agg(F.count("*").alias("n"),
      F.sum("quantity").alias("total_qty"),
      F.round(F.avg("quantity"), 2).alias("avg_qty"))
 .orderBy(F.desc("n"))
 .show())

+---------+-----+
|   status|count|
+---------+-----+
|completed| 4035|
| returned|  503|
|cancelled|  462|
+---------+-----+



+---------+----+---------+-------+
|   status|   n|total_qty|avg_qty|
+---------+----+---------+-------+
|completed|4035|     8905|   2.21|
| returned| 503|     1149|   2.28|
|cancelled| 462|     1027|   2.22|
+---------+----+---------+-------+



In [6]:
# group by several columns, including a computed one
(orders
 .groupBy(F.month("order_ts").alias("month"), "status")
 .count()
 .orderBy("month", "status")
 .show(6))

+-----+---------+-----+
|month|   status|count|
+-----+---------+-----+
|    1|cancelled|   38|
|    1|completed|  327|
|    1| returned|   39|
|    2|cancelled|   37|
|    2|completed|  343|
|    2| returned|   46|
+-----+---------+-----+
only showing top 6 rows


Shortcuts like `.groupBy('status').sum('quantity')` exist but give ugly column names like `sum(quantity)` — prefer `agg` + `alias`.

## Filtering after aggregating (SQL's `HAVING`)
Just `filter` the aggregated DataFrame:

In [7]:
(orders.groupBy("customer_id").agg(F.count("*").alias("n_orders"))
       .filter(F.col("n_orders") > 38)
       .orderBy(F.desc("n_orders"))
       .show())

+-----------+--------+
|customer_id|n_orders|
+-----------+--------+
|        138|      40|
|         68|      39|
+-----------+--------+



## Collecting values into lists

In [8]:
(orders.filter(F.col("customer_id") == 1)
       .groupBy("customer_id")
       .agg(F.collect_list("product_id").alias("all_products"),
            F.sort_array(F.collect_set("product_id")).alias("unique_products"))
       .show(truncate=False))

+-----------+------------------------------------------------------------------------------+----------------------------------------------------------+
|customer_id|all_products                                                                  |unique_products                                           |
+-----------+------------------------------------------------------------------------------+----------------------------------------------------------+
|1          |[25, 11, 17, 20, 4, 5, 4, 24, 28, 4, 18, 3, 28, 3, 8, 22, 6, 7, 23, 30, 18, 5]|[3, 4, 5, 6, 7, 8, 11, 17, 18, 20, 22, 23, 24, 25, 28, 30]|
+-----------+------------------------------------------------------------------------------+----------------------------------------------------------+



## Pivot
Turns the *values* of one column into *columns*. Passing the list of values (`pivot("status", [...])`)
saves Spark an extra pass to discover them.

In [9]:
(orders
 .groupBy(F.month("order_ts").alias("month"))
 .pivot("status", ["completed", "cancelled", "returned"])
 .count()
 .orderBy("month")
 .show())

+-----+---------+---------+--------+
|month|completed|cancelled|returned|
+-----+---------+---------+--------+
|    1|      327|       38|      39|
|    2|      343|       37|      46|
|    3|      318|       29|      42|
|    4|      322|       44|      48|
|    5|      360|       34|      48|
|    6|      320|       41|      36|
|    7|      306|       33|      38|
|    8|      349|       37|      43|
|    9|      342|       44|      45|
|   10|      363|       53|      35|
|   11|      348|       38|      35|
|   12|      337|       34|      48|
+-----+---------+---------+--------+



## Subtotals with `rollup` and `cube`
`rollup("a", "b")` gives groups (a, b), subtotals per a, and a grand total (shown as nulls).

In [10]:
(orders.rollup("status", F.year("order_ts").alias("year"))
       .agg(F.sum("quantity").alias("qty"))
       .orderBy("status", "year")
       .show())

+---------+----+-----+
|   status|year|  qty|
+---------+----+-----+
|     NULL|NULL|11081|
|cancelled|NULL| 1027|
|cancelled|2024| 1027|
|completed|NULL| 8905|
|completed|2024| 8905|
| returned|NULL| 1149|
| returned|2024| 1149|
+---------+----+-----+



## Approximate counts on big data
`countDistinct` is exact but expensive at scale. `approx_count_distinct` (HyperLogLog) is far cheaper and
usually within a couple of percent.

In [11]:
orders.agg(F.countDistinct("customer_id"), F.approx_count_distinct("customer_id")).show()

+---------------------------+----------------------------------+
|count(DISTINCT customer_id)|approx_count_distinct(customer_id)|
+---------------------------+----------------------------------+
|                        180|                               176|
+---------------------------+----------------------------------+



### Exercise 1
Create `qty_by_status` with columns **`status`, `total_qty`** (the sum of quantity per status).

In [12]:
qty_by_status = orders.groupBy("status").agg(F.sum("quantity").alias("total_qty"))

In [13]:
# Check — run me after your answer
assert sorted(qty_by_status.columns) == ["status", "total_qty"]
assert qty_by_status.filter(F.col("status") == "completed").first().total_qty == 8905
print("Correct!")

Correct!


### Exercise 2
Create `top_customers`: the **5** customers with the most orders, columns **`customer_id`, `n_orders`**, sorted by `n_orders` descending.

In [14]:
top_customers = (orders.groupBy("customer_id")
                       .agg(F.count("*").alias("n_orders"))
                       .orderBy(F.desc("n_orders"))
                       .limit(5))

In [15]:
# Check — run me after your answer
assert top_customers.columns == ["customer_id", "n_orders"]
assert [r.n_orders for r in top_customers.collect()] == [40, 39, 38, 38, 38]
print("Correct!")

Correct!


### Exercise 3
Create `monthly` with one row per month (1–12) and columns **`month`, `n_orders`, `n_customers`** (distinct customers), sorted by month.

In [16]:
monthly = (orders.groupBy(F.month("order_ts").alias("month"))
                 .agg(F.count("*").alias("n_orders"),
                      F.countDistinct("customer_id").alias("n_customers"))
                 .orderBy("month"))

In [17]:
# Check — run me after your answer
rows = monthly.collect()
assert monthly.columns == ["month", "n_orders", "n_customers"]
assert len(rows) == 12 and rows[0].month == 1
assert (rows[0].n_orders, rows[0].n_customers) == (404, 157)
assert sum(r.n_orders for r in rows) == 5000
print("Correct!")

Correct!


### Exercise 4
Create `status_pivot`: one row per month with the order **count** for each status as its own column (`completed`, `cancelled`, `returned`).

In [18]:
status_pivot = (orders.groupBy(F.month("order_ts").alias("month"))
                      .pivot("status")
                      .count())

In [19]:
# Check — run me after your answer
assert status_pivot.count() == 12
assert {"cancelled", "completed", "returned"} <= set(status_pivot.columns)
assert status_pivot.agg(F.sum("completed")).first()[0] == 4035
print("Correct!")

Correct!
